In [3]:
import mujoco
import mujoco.viewer


model = mujoco.MjModel.from_xml_path('go2luna.xml')
data = mujoco.MjData(model)

# Create a viewer window and render the model
mujoco.viewer.launch(model, data)

KeyboardInterrupt: 

In [8]:
import numpy as np

def quaternion_multiply(q1, q2):
    """Multiply two quaternions (w, x, y, z format)"""
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    
    w = w1*w2 - x1*x2 - y1*y2 - z1*z2
    x = w1*x2 + x1*w2 + y1*z2 - z1*y2
    y = w1*y2 - x1*z2 + y1*w2 + z1*x2
    z = w1*z2 + x1*y2 - y1*x2 + z1*w2
    
    return [w, x, y, z]

# Your original quaternion (w, x, y, z)
original = [-0.6175196090518028, -0.6175196091784583, -0.3444843629201, -0.344484362849445]

# 90-degree Z-rotation quaternion
z_rot_90 = [ 0.525322 , 0, 0, 0.8509035]

# Apply rotation
result = quaternion_multiply(z_rot_90, original)
print(f"Result: {result}")

Result: [-0.0312736860224484, -0.03127368602886271, -0.706414811266495, -0.7064148111216069]


In [11]:
# Replace 'your_model.xml' with the path to your MuJoCo XML file
model = mujoco.MjModel.from_xml_path('luna.xml')
data = mujoco.MjData(model)

# Create a viewer window and render the model
mujoco.viewer.launch(model, data)

AttributeError: module 'mujoco' has no attribute 'viewer'

In [ ]:
import mujoco
import numpy as np

# Load your model
model = mujoco.MjModel.from_xml_path("go2luna.xml")
data = mujoco.MjData(model)

# Set robot to home position
home_qpos = np.array([0, 0, 0.27, 1, 0, 0, 0, 0, 0.9, -1.8, 0, 0.9, -1.8, 0, 0.9, -1.8, 0, 0.9, -1.8])
data.qpos[:] = home_qpos
mujoco.mj_forward(model, data)

# Calculate total mass
total_mass = np.sum(model.body_mass[1:])  # Exclude world body (index 0)
print(f"Total Mass: {total_mass} kg")

# Calculate Center of Mass in world frame
# MuJoCo stores subtree COM for each body
subtree_com = np.zeros(3)
for i in range(1, model.nbody):  # Skip world body
    body_mass = model.body_subtreemass[i]
    body_com = data.subtree_com[i]
    subtree_com += body_mass * body_com

com_world = subtree_com / total_mass
print(f"Center of Mass (world frame): {com_world}")
print(f"Center of Mass (body frame): {subtree_com}")

# Calculate composite inertia matrix about CoM
# This requires more complex calculation - use MuJoCo's built-in function
M = np.zeros((model.nv, model.nv))
mujoco.mj_fullM(model, M, data)

# Extract the 6x6 base inertia matrix (first 6 DOF for floating base)
base_inertia_6x6 = M[:6, :6]
print(f"Base 6x6 Inertia Matrix:\n{base_inertia_6x6}")

# Extract just the rotational inertia (3x3)
rotational_inertia = base_inertia_6x6[3:6, 3:6]
print(f"Rotational Inertia (3x3):\n{rotational_inertia}")


Total Mass: 12.206408 kg
Center of Mass (world frame): [-5.45678198e-02  4.54770570e-18  5.10018743e-01]
Center of Mass (body frame): [-6.66077072e-01  5.55111512e-17  6.22549687e+00]


TypeError: mj_fullM(): incompatible function arguments. The following argument types are supported:
    1. (m: mujoco._structs.MjModel, dst: typing.Annotated[numpy.typing.NDArray[numpy.float64], "[m, n]", "flags.writeable", "flags.c_contiguous"], M: typing.Annotated[numpy.typing.NDArray[numpy.float64], "[m, 1]"]) -> None

Invoked with: <mujoco._structs.MjModel object at 0x741db0dd1eb0>, [18, 18], <mujoco._structs.MjData object at 0x741db0c17cb0>